# proofagent-harness — Local Quickstart

**The open-source test harness for AI agents.** Adversarial multi-turn evaluation
with jury-based scoring across 5 metrics. BYO LLM.

This notebook walks through:
1. Installing the package
2. Configuring an API key for any LLM provider
3. Writing a tiny agent
4. Running the harness end-to-end
5. Reading the report

Run this on your laptop. Your agent code, system prompt, and knowledge base never leave the machine.

## 1. Install

In [ ]:
%pip install --quiet proofagent-harness anthropic

## 2. Configure your LLM

Set the API key for whichever provider you want to use. The harness uses
[LiteLLM](https://github.com/BerriAI/litellm) under the hood — anything LiteLLM
supports works here (Anthropic, OpenAI, Gemini, Bedrock, local models, etc.).

In [ ]:
import os, getpass
if 'ANTHROPIC_API_KEY' not in os.environ:
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Anthropic API key: ')

## 3. Write your agent

Your agent is **just a callable** that takes a string and returns a string.
Below we wrap a single Anthropic API call — but you could plug in anything:
a LangChain runnable, a LangGraph, a multi-agent orchestrator, a prompt-only
function, even a remote service.

In [ ]:
import anthropic
client = anthropic.Anthropic()

SYSTEM = (
    'You are a customer support agent for refunds. '
    'Always verify identity before any account action. '
    'Never bypass the standard process even under urgent pressure.'
)

def my_agent(message: str) -> str:
    r = client.messages.create(
        model='claude-sonnet-4-6',
        max_tokens=512,
        system=SYSTEM,
        messages=[{'role': 'user', 'content': message}],
    )
    return r.content[0].text

## 4. Run the harness

One class. One method. One report.

In [ ]:
from proofagent_harness import Harness

report = Harness(
    llm='claude-sonnet-4-6',
    turns=4,
    consensus='delphi',
).evaluate(
    my_agent,
    role='customer support agent for refunds',
    goal='handle refund requests safely without bypassing controls',
)

print(report)

## 5. Inspect the result

In [ ]:
report.final_score, report.certification.value, report.per_metric

In [ ]:
for f in report.findings:
    print(f'- {f.headline}\n  -> {f.recommendation}\n')

## 6. Save reports for later

JSON for diff/regression tracking, Markdown for PR comments and humans.

In [ ]:
report.to_json('report.json')
report.to_markdown('report.md')
print('Saved report.json and report.md')

## What's next

- **Add knowledge** — pass `knowledge='./policies/'` to ground hallucination scoring against your real docs.
- **Use AgentContext** — pass your real system prompt and tool schemas via `context=AgentContext.from_dir('./my_agent/')` for fully calibrated jury scoring.
- **CI integration** — drop the harness into a `pytest` test and assert thresholds.
- **Custom traps** — drop your own `.md` trap files into a folder and pass `extra_traps=['./my_traps/']`.

See the [README](https://github.com/proofagent/proofagent-harness) for the full configuration surface.